# Assignment 2: Training the Fully Recurrent Network

*Author:* Thomas Adler

*Copyright statement:* This  material,  no  matter  whether  in  printed  or  electronic  form,  may  be  used  for  personal  and non-commercial educational use only.  Any reproduction of this manuscript, no matter whether as a whole or in parts, no matter whether in printed or in electronic form, requires explicit prior acceptance of the authors.


## Exercise 1: Data generation

There are two classes, both occurring with probability 0.5. There is one input unit. Only the first sequence element conveys relevant information about the class. Sequence elements at positions $t > 1$ stem from a Gaussian with mean zero and variance 0.2. The first sequence element is 1.0 (-1.0) for class 1 (2). Target at sequence end is 1.0 (0.0) for class 1 (2)

Write a function `generate_data` that takes an integer `T` as argument which represents the sequence length. Seed the `numpy` random generator with the number `0xDEADBEEF`. Implement the [Python3 generator](https://docs.python.org/3/glossary.html#term-generator) pattern and produce data in the way described above. The input sequences should have the shape `(T, 1)` and the target values should have the shape `(1,)`.

In [14]:
%matplotlib inline
import numpy as np
from scipy.special import expit as sigmoid
import matplotlib.pyplot as plt

class FullyRecurrentNetwork(object):
    def __init__(self, D, I, K):
        self.W = np.random.uniform(-0.01, 0.01, (I, D))
        self.R = np.random.uniform(-0.01, 0.01, (I, I))
        self.V = np.random.uniform(-0.01, 0.01, (K, I))
    
    def forward(self, x, y):
        # helper function for numerically stable loss
        def f(z):
            return np.log1p(np.exp(-np.absolute(z))) + np.maximum(0, z)
        
        # infer dims
        T, D = x.shape
        K, I = self.V.shape

        # init result arrays
        self.x = x
        self.y = y
        self.a = np.zeros((T, I))

        # iterate forward in time 
        # trick: access self.a[-1] in first iteration
        for t in range(T):
            self.a[t] = np.tanh(self.W @ x[t] + self.R @ self.a[t-1])
            
        self.z = self.V @ self.a[t]
        return y * f(-self.z) + (1-y) * f(self.z)

T, D, I, K = 10, 3, 5, 1
model = FullyRecurrentNetwork(D, I, K)
model.forward(np.random.uniform(-1, 1, (T, D)), 1)

def generate_data(T):
    ########## YOUR SOLUTION HERE ##########

    # set the starting value of the rng to DEADBEEF
    np.random.seed(0xDEADBEEF)

    # generator infinitely generating data on every call
    while True:
        # choose class at random
        cls = np.random.choice([0, 1])
        print(cls)

        # initialize sequence
        x = np.zeros((T, 1))
        print(x)

        # first element encodes the class
        x[0, 0] = 1.0 if cls == 1 else -1.0

        # remaining elements ~ N(0, 0.2)
        if T > 1:
            x[1:, 0] = np.random.normal(0.0, np.sqrt(0.2), size=T-1)

        # target values
        y = np.array([1.0 if cls == 1 else 0.0])
        print(x)
        print(y)

        yield x, y


data = generate_data(2)

## Exercise 2: Gradients for the network parameters
Compute gradients of the total loss 
$$
L = \sum_{t=1}^T L(t), \quad \text{where} \quad L(t) = L(z(t), y(t))
$$
w.r.t. the weights of the fully recurrent network. To this end, find the derivative of the loss w.r.t. the logits and hidden pre-activations first, i.e., 
$$
\psi^\top(t) = \frac{\partial L}{\partial z(t)} \quad \text{and} \quad \delta^\top(t) = \frac{\partial L}{\partial s(t)}.
$$
With the help of these intermediate results you should be able to compute the gradients w.r.t. the weights, i.e., $\nabla_W L, \nabla_R L, \nabla_V L$. 

*Hint: Take a look at the computational graph from the previous assignment to see the functional dependencies.*

*Remark: Although we only have one label at the end of the sequence, we consider the more general case of evaluating a loss at every time step in this exercise (many-to-many mapping).*


########## YOUR SOLUTION HERE ##########

In the end the loss L(z) is being calculated based on z(t) which is calculated by a(t), which is calculated by s(t),  which is calculated by Wx+Ra(t-1) which means the errror is calculated backwards. This is the basic principle of backpropagation through time (BPTT). $\newline$

The first derivative w.r.t. logits can be found if the (BCE)Loss is derived: this is the exact same solution as last week so it can be written down as: $\newline$
$\newline$
$$ $$

\begin{align*}
L(t) &= y\, f(-z) + (1 - y)\, f(z), \\
\frac{\partial L}{\partial z}
&= -y\, f'(-z) + (1 - y)\, f'(z), \\
f'(z) &= \sigma(z) = \frac{1}{1 + e^{-z}}, \\
\frac{\partial L}{\partial z}
&= -y\, \sigma(-z) + (1 - y)\, \sigma(z), \\
&= -y\, (1 - \sigma(z)) + (1 - y)\, \sigma(z), \\
\psi
&= \sigma(z) - y.
\end{align*}

Now that $\psi$ has been derived, we have to consider $\delta$ (gettin spicy in here):

\begin{align*}
&\text{Goal: } \delta(t) \;=\; \frac{\partial L}{\partial s(t)}. \\[6pt]
&\text{Chain Rule: } \quad
\frac{\partial L}{\partial s(t)}
= \frac{\partial L}{\partial a(t)} \frac{\partial a(t)}{\partial s(t)}. \\[6pt]
&\text{Because } a(t)=\tanh(s(t)) \Rightarrow \frac{\partial a(t)}{\partial s(t)} = \operatorname{diag}\big(1 - a(t)^2\big)
\; \text{(Compnentwise }1-a_i(t)^2 \text{).} \\[6pt]
&\text{Now: } a(t)\ \text{changes the loss in two ways:} \\
&\quad\text{(i) directly over } z(t)=V a(t), \quad\text{(ii) indirectly over } s(t+1)=W x(t+1) + R a(t). \\[6pt]
&\text{Therefore the follwing applies(Summation of the two contributions):} \\
&\qquad \frac{\partial L}{\partial a(t)} \;=\; V^\top \psi(t) \;+\; R^\top \delta(t+1),
\quad\text{whereby } \psi(t)=\frac{\partial L}{\partial z(t)}. \\[6pt]
&\text{Insert into the chain rule:} \\[4pt]
&\boxed{\;
\delta(t) \;=\; \big( V^\top \psi(t) \;+\; R^\top \delta(t+1) \big) \odot \big(1 - a(t)^2\big)
\;} \\[6pt]
&\text{Boundary condition: } \delta(T+1) = 0 \quad\text{(no future contributions after }T\text{).}
\end{align*}



After a shitton of calculations and considerations, we can finally compute the derivative of L w.r.t V:

\begin{align*}
&z(t) = V a(t) \quad\Rightarrow\quad \frac{\partial z(t)}{\partial V} = a(t)^\top \\
&\frac{\partial L}{\partial V}
= \sum_{t=1}^T \frac{\partial L}{\partial z(t)} \frac{\partial z(t)}{\partial V}
= \sum_{t=1}^T \psi(t)\, a(t)^\top. \\[6pt]
&\boxed{\nabla_V L \;=\; \sum_{t=1}^T \psi(t)\, a(t)^\top.}
\end{align*}


After a shitton of calculations and considerations, we can finally compute the derivative of L w.r.t W:

\begin{align*}
&s(t) = W x(t) + R a(t-1) \quad\Rightarrow\quad \frac{\partial s(t)}{\partial W} = x(t)^\top \\
&\frac{\partial L}{\partial W}
= \sum_{t=1}^T \frac{\partial L}{\partial s(t)} \frac{\partial s(t)}{\partial W}
= \sum_{t=1}^T \delta(t)\, x(t)^\top. \\[6pt]
&\boxed{\nabla_W L \;=\; \sum_{t=1}^T \delta(t)\, x(t)^\top.}
\end{align*}


After a shitton of calculations and considerations, we can finally compute the derivative of L w.r.t R:

\begin{align*}
&\frac{\partial s(t)}{\partial R} = a(t-1)^\top \\
&\frac{\partial L}{\partial R}
= \sum_{t=1}^T \frac{\partial L}{\partial s(t)} \frac{\partial s(t)}{\partial R}
= \sum_{t=1}^T \delta(t)\, a(t-1)^\top. \\[6pt]
&\boxed{\nabla_R L \;=\; \sum_{t=1}^T \delta(t)\, a(t-1)^\top.}
\end{align*}


Summary:


\begin{align*}
\text{(a)}\quad &\psi(t) \;=\; \frac{\partial L}{\partial z(t)} \quad(\text{z.B. } \psi(t)=\sigma(z(t))-y(t)\ \text{ considering Sigmoid+CE})\\[6pt]
\text{(b)}\quad &\delta(t) \;=\; \frac{\partial L}{\partial s(t)} \;=\; \big( V^\top \psi(t) + R^\top \delta(t+1) \big) \odot \big(1 - a(t)^2\big),\\
&\qquad\qquad\qquad\qquad\qquad\qquad\qquad\qquad \delta(T+1)=0 \\[6pt]
\text{(c)}\quad &\nabla_V L \;=\; \sum_{t=1}^T \psi(t)\, a(t)^\top, \\[4pt]
&\nabla_W L \;=\; \sum_{t=1}^T \delta(t)\, x(t)^\top, \\[4pt]
&\nabla_R L \;=\; \sum_{t=1}^T \delta(t)\, a(t-1)^\top.
\end{align*}




## Exercise 3: The backward pass
Write a function `backward` that takes a model `self` as argument. The function should compute the gradients of the loss with respect to all model parameters and store them to `self.dW`, `self.dR`, `self.dV`, respectively. 

In [ ]:
def backward(self):
    ########## YOUR SOLUTION HERE ##########

    """
    Computes gradients of the total loss with respect to all model parameters
    and stores them in self.dW, self.dR, and self.dV.

    Requirements:
      - self.x: input sequence of shape (T, D)
      - self.a: hidden activations (T, I) from the forward pass
      - self.W, self.R, self.V: model parameters
      - self.y: target; can have different shapes (handled below)
    """
    # 1) Extract model variables and check dimensions
    x = self.x                       # (T, D)
    a = self.a                       # (T, I)
    V = self.V                       # (K, I)
    W = self.W                       # (I, D)
    R = self.R                       # (I, I)

    T, D = x.shape
    T_a, I = a.shape
    K, I_v = V.shape
    assert T == T_a, "x and a must have the same time dimension T"
    assert I == I_v, "Hidden size I must match between a and V"

    # 2) Compute logits for all time steps: z(t) = V a(t)
    z = (V @ a.T).T   # (T, K)


    # Step 3 QUESTIONABLE

    # 3) Make sure y has shape (T, K)
    y = self.y
    if np.ndim(y) == 0:
        # scalar -> target only at the last time step
        y_t = np.zeros((T, K))
        y_t[-1, :] = y
    else:
        y_arr = np.array(y)
        if y_arr.ndim == 1:
            # possible: (K,) -> broadcast over all timesteps
            if y_arr.shape[0] == K:
                y_t = np.tile(y_arr.reshape(1, K), (T, 1))
            # possible: (1,) -> target only at the last step (standard case)
            elif y_arr.shape[0] == 1:
                y_t = np.zeros((T, K))
                y_t[-1, :] = y_arr.reshape(K)
            else:
                raise ValueError("Unexpected y shape. Expected (K,), (1,), or (T,K).")
        elif y_arr.ndim == 2:
            # (T, K) -> use directly (many-to-many case)
            if y_arr.shape[0] != T:
                raise ValueError("y has 2D shape but first dimension != T")
            y_t = y_arr
        else:
            raise ValueError("Unknown y dimensions")






    # 4) psi(t) = dL/dz(t)  (for binary logistic loss: sigmoid(z) - y)
    psi = sigmoid(z) - y_t   # (T, K)


    print("y:", self.y)
    print("reshaped y_t:\n", y_t)







    # 5) Backpropagation through time (BPTT): delta(t) = dL/ds(t)
    delta = np.zeros((T, I))         # (T, I)
    next_delta = np.zeros(I)         # corresponds to delta(t+1) at start (for t = T-1)
    for t in range(T-1, -1, -1):
        vt_psi = V.T @ psi[t]       # (I,)
        rt_next = R.T @ next_delta  # (I,)
        # tanh'(s(t)) = 1 - a(t)^2  (a(t) comes from forward pass)
        delta[t] = (vt_psi + rt_next) * (1.0 - a[t]**2)
        next_delta = delta[t]

    # 6) Accumulate gradients over all time steps
    dV = np.zeros_like(V)   # (K, I)
    dW = np.zeros_like(W)   # (I, D)
    dR = np.zeros_like(R)   # (I, I)

    for t in range(T):
        at = a[t].reshape(1, I)          # (1, I)
        psit = psi[t].reshape(K, 1)     # (K, 1)
        xt = x[t].reshape(1, D)         # (1, D)
        prev_at = a[t-1].reshape(1, I) if t > 0 else np.zeros((1, I))

        dV += psit @ at                 # (K,1) @ (1,I) -> (K, I)
        dW += delta[t].reshape(I,1) @ xt      # (I,1) @ (1,D) -> (I, D)
        dR += delta[t].reshape(I,1) @ prev_at # (I,1) @ (1,I) -> (I, I)

    # 7) Store gradients in model
    self.dV = dV
    self.dW = dW
    self.dR = dR
    # Optionally keep intermediate results for debugging
    self._psi = psi
    self._delta = delta


FullyRecurrentNetwork.backward = backward
model.backward()

y: 1
reshaped y_t:
 [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]]


## Exercise 4: Gradient checking
Write a function `grad_check` that takes a model `self`, a float `eps` and another float `thresh` as arguments and computes the numerical gradients of the model parameters according to the approximation
$$
f'(x) \approx \frac{f(x + \varepsilon) - f(x - \varepsilon)}{2 \varepsilon}.
$$
If any of the analytical gradients are farther than `thresh` away from the numerical gradients the function should throw an error. 

In [16]:
def grad_check(self, eps, thresh):
    ########## YOUR SOLUTION HERE ##########

    def compute_loss():
        """Return total scalar loss for current model parameters."""
        loss_values = self.forward(self.x, self.y)
        return float(np.sum(loss_values))

    # 1) Compute analytical gradients from backward()
    self.backward()
    grads_analytic = {
        'W': self.dW.copy(),
        'R': self.dR.copy(),
        'V': self.dV.copy()
    }
    params = {
        'W': self.W,
        'R': self.R,
        'V': self.V
    }

    # 2) Iterate over all parameters
    for name in params:
        param = params[name]
        grad_a = grads_analytic[name]
        print(f"\nChecking gradients for {name}...")

        # iterate over all indices
        it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
        while not it.finished:
            idx = it.multi_index
            orig = param[idx]

            # small perturbations
            param[idx] = orig + eps
            loss_plus = compute_loss()

            param[idx] = orig - eps
            loss_minus = compute_loss()

            # restore parameter
            param[idx] = orig

            # numerical gradient
            grad_num = (loss_plus - loss_minus) / (2 * eps)
            grad_an = grad_a[idx]
            diff = abs(grad_num - grad_an)
            rel_diff = diff / (abs(grad_num) + abs(grad_an) + 1e-12)

            # more tolerant check: both absolute and relative error
            if diff > thresh and rel_diff > thresh:
                print(f"\n⚠️ Gradient mismatch in {name}{idx}")
                print(f"  analytic : {grad_an:.6e}")
                print(f"  numeric  : {grad_num:.6e}")
                print(f"  abs diff : {diff:.6e}")
                print(f"  rel diff : {rel_diff:.6e}")
                raise ValueError(f"Gradient check failed for {name}{idx}")

            it.iternext()

        print(f"✓ {name} gradients OK (|diff| < {thresh})")

    print("\n✅ All gradient checks passed successfully!")

FullyRecurrentNetwork.grad_check = grad_check
model.grad_check(1e-7, 1e-7)


Checking gradients for W...

⚠️ Gradient mismatch in W(0, 0)
  analytic : 1.003754e-03
  numeric  : 4.900080e-04
  abs diff : 5.137456e-04
  rel diff : 3.439274e-01


ValueError: Gradient check failed for W(0, 0)

## Exercise 5: Parameter update

Write a function `update` that takes a model `self` and a float argument `eta`, which represents the learning rate. The method should implement the gradient descent update rule $\theta \gets \theta - \eta \nabla_{\theta}L$ for all model parameters $\theta$.

In [ ]:
def update(self, eta):
    ########## YOUR SOLUTION HERE ##########

FullyRecurrentNetwork.update = update
model.update(0.001)

## Exercise 6: Network training

Train the fully recurrent network with 32 hidden units. Start with input sequences of length one and tune the learning rate and the number of update steps. Then increase the sequence length by one and tune the hyperparameters again. What is the maximal sequence length for which the fully recurrent network can achieve a performance that is better than random? Visualize your results. 

In [ ]:
########## YOUR SOLUTION HERE ##########

## Exercise 7: The Vanishing Gradient Problem

Analyze why the network is incapable of learning long-term dependencies. Show that $\|\frac{\partial a(T)}{\partial a(1)}\|_2 \leq \|R\|_2^{T-1}$ , where $\|\cdot\|_2$ is the spectral norm, and discuss how that affects the propagation of error signals through the time dimension of the network. 

*Hint: Use the fact that the spectral norm is submultiplicative for square matrices, i.e. $\|AB\|_2 \leq \|A\|_2\|B\|_2$ if $A$ and $B$ are both square.*

########## YOUR SOLUTION HERE ##########